[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_13/NB_bishop_ch13_contagion.ipynb)

# Chapter 13: Interbank Contagion Modelling

**Bishop & Bishop (2024), Chapter 13**

We model an interbank network where weighted edges represent credit exposures.
When a bank defaults, losses cascade through the network. We run Monte Carlo
simulations to quantify systemic risk and identify systemically important banks.

**Course:** Neural Networks and Deep Learning with Business Applications

**Author:** Daniel Traian PELE -- Bucharest University of Economic Studies

In [ ]:
# Run in Google Colab to install dependencies
!pip install -q networkx


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx

# Course colors
BLUE    = '#1f4e79'
CRIMSON = '#c00000'
GREEN   = '#2e7d32'

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor']   = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['figure.figsize'] = (10, 5)

np.random.seed(42)

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Chapter 13: Interbank Contagion Modelling -- Ready!')

## 1. Create an Interbank Network

We model **20 banks** connected by interbank lending.

- **Directed, weighted edges**: edge $(i \to j)$ with weight $w_{ij}$ means
  bank $j$ has lent $w_{ij}$ to bank $i$ (i.e., $j$ is exposed to $i$).
- Each bank has a **capital buffer** that absorbs losses before default.
- Bank sizes follow a power-law: a few large banks, many small ones.

In [ ]:
np.random.seed(42)

N_BANKS = 20
bank_names = [f'Bank {i}' for i in range(N_BANKS)]

# Total assets follow a power-law distribution (some banks are much larger)
raw_sizes = np.random.pareto(a=1.5, size=N_BANKS) + 1
total_assets = (raw_sizes / raw_sizes.sum() * 5000).round(0)  # in billions EUR
total_assets = np.sort(total_assets)[::-1]  # largest first

# Capital buffer = 5-12% of total assets
capital_ratio = np.random.uniform(0.05, 0.12, size=N_BANKS)
capital_buffer = (total_assets * capital_ratio).round(1)

# Create directed graph with random interbank exposures
G = nx.DiGraph()
for i in range(N_BANKS):
    G.add_node(i, name=bank_names[i], assets=total_assets[i],
               capital=capital_buffer[i])

# Each bank lends to 3-8 other banks; exposure proportional to borrower size
for i in range(N_BANKS):
    n_counterparties = np.random.randint(3, min(9, N_BANKS))
    partners = np.random.choice(
        [j for j in range(N_BANKS) if j != i],
        size=n_counterparties, replace=False
    )
    for j in partners:
        # Exposure from i to j (j is creditor of i)
        exposure = np.random.uniform(0.5, 3.0) * total_assets[i] * 0.05
        if not G.has_edge(i, j):
            G.add_edge(i, j, weight=round(exposure, 1))

print(f'Interbank network: {G.number_of_nodes()} banks, {G.number_of_edges()} exposures')

# Summary table
bank_df = pd.DataFrame({
    'Bank':    bank_names,
    'Assets':  total_assets,
    'Capital': capital_buffer,
    'Cap%':    (capital_ratio * 100).round(1),
    'Out-degree': [G.out_degree(i) for i in range(N_BANKS)],
    'In-degree':  [G.in_degree(i)  for i in range(N_BANKS)],
})
print('\nBank summary (top 10 by assets):')
bank_df.head(10)

In [ ]:
# Visualize the interbank network
pos = nx.spring_layout(G, seed=42, k=1.5)

# Node sizes proportional to assets
node_sizes = (total_assets / total_assets.max() * 800 + 200).astype(int)

# Edge widths proportional to exposure
edge_weights = np.array([G[u][v]['weight'] for u, v in G.edges()])
edge_widths = (edge_weights / edge_weights.max() * 3 + 0.3)

fig, ax = plt.subplots(figsize=(11, 9))
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color=BLUE, alpha=0.85)
nx.draw_networkx_labels(G, pos, ax=ax,
                        labels={i: f'{i}' for i in range(N_BANKS)},
                        font_size=8, font_color='white')
nx.draw_networkx_edges(G, pos, ax=ax, width=edge_widths,
                       edge_color='gray', alpha=0.5,
                       arrows=True, arrowsize=10,
                       connectionstyle='arc3,rad=0.1')
ax.set_title('Interbank Exposure Network\n(node size = total assets, edge width = exposure)',
             fontsize=13)
fig.tight_layout()
save_fig(fig, 'bishop_ch13_interbank_network')
plt.show()

## 2. Contagion Simulation Engine

**Cascade mechanism:**

1. An initial bank defaults (exogenous shock).
2. All banks that lent to the defaulted bank suffer a loss equal to their exposure
   (assuming zero recovery).
3. If a bank's accumulated losses exceed its capital buffer, it defaults too.
4. Repeat until no new defaults occur.

This is the **Eisenberg-Noe** style clearing mechanism simplified for illustration.

In [ ]:
def simulate_contagion(G, initial_defaults, capital_buffer, recovery_rate=0.0):
    """
    Simulate a default cascade on the interbank network.
    
    Parameters
    ----------
    G : nx.DiGraph
        Interbank network. Edge (i -> j) means j is exposed to i.
    initial_defaults : list of int
        Banks that default exogenously.
    capital_buffer : array (N,)
        Capital buffer for each bank.
    recovery_rate : float
        Fraction of exposure recovered (0 = total loss).
    
    Returns
    -------
    defaulted : set of int
        All banks that defaulted (including initial).
    losses : array (N,)
        Total losses incurred by each bank.
    rounds : int
        Number of cascade rounds.
    """
    N = G.number_of_nodes()
    losses = np.zeros(N)
    defaulted = set(initial_defaults)
    new_defaults = set(initial_defaults)
    rounds = 0
    
    while new_defaults:
        rounds += 1
        next_defaults = set()
        
        for d in new_defaults:
            # Find all creditors of the defaulted bank
            # Edge (d -> j) means j is a creditor exposed to d
            for _, creditor, data in G.out_edges(d, data=True):
                if creditor not in defaulted:
                    loss = data['weight'] * (1 - recovery_rate)
                    losses[creditor] += loss
                    
                    # Check if creditor defaults
                    if losses[creditor] >= capital_buffer[creditor]:
                        next_defaults.add(creditor)
        
        defaulted.update(next_defaults)
        new_defaults = next_defaults
    
    return defaulted, losses, rounds


# Test: what happens if Bank 0 (largest) defaults?
defaulted, losses, rounds = simulate_contagion(G, [0], capital_buffer)
print(f'Initial default: Bank 0 (assets = {total_assets[0]:.0f} bn)')
print(f'Cascade rounds:  {rounds}')
print(f'Total defaults:  {len(defaulted)} / {N_BANKS}')
print(f'Defaulted banks: {sorted(defaulted)}')
print(f'Total losses:    {losses.sum():.1f} bn EUR')

In [ ]:
# Single-bank contagion impact for each bank
impact = []
for i in range(N_BANKS):
    defs, loss, rnds = simulate_contagion(G, [i], capital_buffer)
    impact.append({
        'bank': i,
        'assets': total_assets[i],
        'capital': capital_buffer[i],
        'n_defaults': len(defs),
        'cascade_rounds': rnds,
        'total_loss': loss.sum()
    })

impact_df = pd.DataFrame(impact).sort_values('n_defaults', ascending=False)
print('Contagion impact when each bank defaults individually:')
impact_df.head(10)

## 3. Monte Carlo Simulation (100 Scenarios)

In each scenario we randomly select 1--3 banks to default (exogenous shock)
and simulate the cascade. We record total defaults and total system losses.

In [ ]:
np.random.seed(42)

N_SCENARIOS = 100
mc_results = []

for s in range(N_SCENARIOS):
    # Random shock: 1-3 initial defaults
    n_initial = np.random.randint(1, 4)
    initial = list(np.random.choice(N_BANKS, size=n_initial, replace=False))
    
    defs, loss, rnds = simulate_contagion(G, initial, capital_buffer)
    
    mc_results.append({
        'scenario':        s,
        'n_initial':       n_initial,
        'initial_banks':   initial,
        'n_defaults':      len(defs),
        'cascade_defaults': len(defs) - n_initial,
        'cascade_rounds':  rnds,
        'total_loss':      loss.sum(),
        'defaulted_set':   defs
    })

mc_df = pd.DataFrame(mc_results)

print(f'Monte Carlo: {N_SCENARIOS} scenarios')
print(f'\nTotal defaults per scenario:')
print(mc_df['n_defaults'].describe().round(2))
print(f'\nTotal loss per scenario (bn EUR):')
print(mc_df['total_loss'].describe().round(1))

In [ ]:
# Plot: distribution of total defaults and total losses
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Defaults histogram
ax1.hist(mc_df['n_defaults'], bins=range(0, N_BANKS + 2), color=CRIMSON,
         edgecolor='white', alpha=0.85, align='left')
ax1.axvline(mc_df['n_defaults'].mean(), color=BLUE, linestyle='--', linewidth=2,
            label=f'Mean = {mc_df["n_defaults"].mean():.1f}')
ax1.set_xlabel('Total Defaults (incl. initial)', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Distribution of Total Defaults', fontsize=12)
ax1.legend(fontsize=10)

# Losses histogram
ax2.hist(mc_df['total_loss'], bins=25, color=BLUE, edgecolor='white', alpha=0.85)
ax2.axvline(mc_df['total_loss'].mean(), color=CRIMSON, linestyle='--', linewidth=2,
            label=f'Mean = {mc_df["total_loss"].mean():.0f} bn')
p95 = mc_df['total_loss'].quantile(0.95)
ax2.axvline(p95, color=GREEN, linestyle='--', linewidth=2,
            label=f'95th pct = {p95:.0f} bn')
ax2.set_xlabel('Total System Loss (bn EUR)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Distribution of System Losses', fontsize=12)
ax2.legend(fontsize=10)

fig.tight_layout()
save_fig(fig, 'bishop_ch13_contagion_mc')
plt.show()

## 4. Identify Systemically Important Banks

A bank is **systemically important** if its failure triggers many additional
defaults. We measure this by how often each bank appears in the defaulted set
across all Monte Carlo scenarios and by its single-shock cascade size.

In [ ]:
# Count how often each bank defaults across all scenarios
default_freq = np.zeros(N_BANKS)
for row in mc_results:
    for b in row['defaulted_set']:
        default_freq[b] += 1

# Combine with single-bank impact
sysimport_df = pd.DataFrame({
    'Bank':           range(N_BANKS),
    'Assets':         total_assets,
    'Capital':        capital_buffer,
    'Default_freq':   default_freq.astype(int),
    'Single_cascade': [impact[i]['n_defaults'] for i in range(N_BANKS)],
    'Single_loss':    [impact[i]['total_loss']  for i in range(N_BANKS)],
}).sort_values('Single_cascade', ascending=False)

print('Systemically Important Banks (sorted by cascade size):')
sysimport_df.head(10)

In [ ]:
# Visualize: systemic importance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Bar chart: cascade defaults per bank
sorted_impact = impact_df.sort_values('bank')
colors_bar = [CRIMSON if n > 3 else BLUE for n in sorted_impact['n_defaults']]
ax1.bar(range(N_BANKS), sorted_impact['n_defaults'].values, color=colors_bar,
        edgecolor='white')
ax1.set_xlabel('Bank ID', fontsize=11)
ax1.set_ylabel('Total Defaults Triggered', fontsize=11)
ax1.set_title('Single-Bank Contagion Impact', fontsize=12)
ax1.set_xticks(range(N_BANKS))

# Scatter: assets vs cascade loss
ax2.scatter(impact_df['assets'], impact_df['total_loss'],
            s=impact_df['n_defaults'] * 60, c=CRIMSON, alpha=0.7, edgecolors=BLUE)
for _, row in impact_df.iterrows():
    if row['n_defaults'] > 3:
        ax2.annotate(f"Bank {int(row['bank'])}",
                     (row['assets'], row['total_loss']),
                     fontsize=8, ha='left', va='bottom')
ax2.set_xlabel('Bank Assets (bn EUR)', fontsize=11)
ax2.set_ylabel('Cascade Total Loss (bn EUR)', fontsize=11)
ax2.set_title('Assets vs Contagion Loss\n(bubble size = # defaults)', fontsize=12)

fig.tight_layout()
save_fig(fig, 'bishop_ch13_systemic_importance')
plt.show()

In [ ]:
# Visualize a worst-case cascade on the network
worst_idx = mc_df['n_defaults'].idxmax()
worst = mc_results[worst_idx]

node_colors_casc = []
for i in range(N_BANKS):
    if i in worst['initial_banks']:
        node_colors_casc.append('#ff6600')  # orange = initial shock
    elif i in worst['defaulted_set']:
        node_colors_casc.append(CRIMSON)    # red = cascade default
    else:
        node_colors_casc.append(GREEN)      # green = survived

fig, ax = plt.subplots(figsize=(11, 9))
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                       node_color=node_colors_casc, alpha=0.85)
nx.draw_networkx_labels(G, pos, ax=ax, labels={i: str(i) for i in range(N_BANKS)},
                        font_size=8, font_color='white')
nx.draw_networkx_edges(G, pos, ax=ax, width=edge_widths,
                       edge_color='gray', alpha=0.4,
                       arrows=True, arrowsize=10,
                       connectionstyle='arc3,rad=0.1')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff6600', markersize=10,
           label=f'Initial shock ({len(worst["initial_banks"])} banks)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CRIMSON, markersize=10,
           label=f'Cascade default ({worst["cascade_defaults"]} banks)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=GREEN, markersize=10,
           label=f'Survived ({N_BANKS - len(worst["defaulted_set"])} banks)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
ax.set_title(f'Worst-Case Cascade (Scenario {worst_idx}): '
             f'{len(worst["defaulted_set"])} / {N_BANKS} banks default',
             fontsize=13)
fig.tight_layout()
save_fig(fig, 'bishop_ch13_worst_cascade')
plt.show()

## 5. Connection to ECB / BNR Stress Testing

Our simplified contagion model captures the essence of real-world regulatory
stress tests:

| Our Model | ECB / BNR Practice |
|---|---|
| Random initial defaults | Macro-economic adverse scenarios |
| Binary default threshold | Gradual capital depletion + CET1 thresholds |
| Zero recovery rate | Recovery rates calibrated by asset class |
| Cascade rounds | Multi-period dynamic balance sheet simulation |
| Systemically important banks | G-SIB / O-SII designation + capital surcharges |

**Key regulatory frameworks:**

- **ECB SREP** (Supervisory Review and Evaluation Process): assesses bank-level
  and system-level risks, including interconnectedness.
- **EBA EU-wide stress test**: biennial macro stress test covering top-down
  and bottom-up approaches.
- **BNR (National Bank of Romania)** applies ECB/EBA methodology to the Romanian
  banking system, with additional focus on FX exposures and domestic contagion.
- **Graph neural networks** are increasingly used in research to model contagion
  because they naturally encode network topology and can learn non-linear
  propagation dynamics from data.

## Key Takeaways

1. **Contagion cascades** can amplify a single bank failure into a system-wide
   crisis. The interbank network is the transmission channel.

2. **Systemically important banks** are not always the largest -- a
   medium-sized bank with many large exposures can trigger more cascade
   defaults than a larger but less connected bank.

3. **Capital buffers** are the key defence. Higher capital ratios reduce
   the probability that losses propagate to the next layer.

4. **Monte Carlo simulation** quantifies tail risk: the 95th percentile of
   system losses is the relevant metric for regulators.

5. **GNNs for contagion**: message passing in a GNN is structurally analogous
   to loss propagation in a contagion model -- both aggregate information
   from neighbours and transform it. This makes GNNs a natural architecture
   for learning contagion dynamics from historical data.

In [ ]:
takeaways = [
    '1. Contagion cascades amplify single failures into systemic crises via the interbank network.',
    '2. Systemically important banks are defined by connectivity, not just size.',
    '3. Capital buffers are the primary defence against cascade propagation.',
    '4. Monte Carlo simulation quantifies tail risk (95th percentile losses).',
    '5. GNN message passing is structurally analogous to contagion loss propagation.',
]
print('Key Takeaways')
print('=' * 60)
for t in takeaways:
    print(t)